# Agent Repair Pipeline — FEVER

**ICLR Experiment Pipeline — Colab A100-80GB + Qwen2.5-32B-Instruct-AWQ**

| Stage | What it does | ~Time (A100-80GB, 500 Qs) |
|---|---|---|
| 0 | Download FEVER, create folders | 1 min |
| 1 | ReAct trajectory generation (Qwen2.5-32B-Instruct-AWQ) | ~1-2 h |
| 2 | Step-level uncertainty scoring | ~1-2 h |
| 3 | 72B judge error annotation | ~1-2 h |
| 4 | Localization scoring (+ cascade-aware rules) | < 1 min |
| 5 | 126 repair strategies (backtrack × nudge type ablation) | ~4-6 h |
| 6 | Tables, stats, figures, ablation analysis | < 1 min |

**Every stage is resumable** — if Colab disconnects, re-run the cell and it picks up where it left off.

---
## 0. Setup

In [1]:
# Verify GPU — should show A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

NVIDIA A100-SXM4-80GB, 81920 MiB


In [2]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab_fever.yaml'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kishormorol/agent-repair.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

Cloning into '/content/agent-repair'...
remote: Enumerating objects: 214, done.
remote: Counting objects: 100% (214/214), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 214 (delta 107), reused 171 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (214/214), 316.57 KiB | 8.33 MiB/s, done.
Resolving deltas: 100% (107/107), done.
Working directory: /content/agent-repair


In [4]:
# Check Colab's CUDA version first
!nvcc --version 2>/dev/null || echo "nvcc not found"
!python -c "import torch; print(f'torch CUDA: {torch.version.cuda}')" 2>/dev/null || echo "torch not installed yet"
!nvidia-smi | grep "CUDA Version"

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
torch CUDA: 12.8
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |


In [5]:
# Install vLLM — pinned to 0.19.0 (last version with CUDA 12 default wheels)
# vLLM >= 0.20 ships CUDA 13 wheels which don't work on Colab's CUDA 12.x
!pip uninstall -y vllm 2>/dev/null
!pip install -q "vllm==0.19.0" 2>&1 | tail -5

# Install remaining dependencies
!pip install -q "transformers>=4.57.0" "tokenizers>=0.21.0" accelerate \
    huggingface_hub datasets scipy scikit-learn pandas pyarrow \
    pyyaml tqdm matplotlib seaborn statsmodels 2>&1 | tail -3

print('\n--- Installation complete ---')

gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.

--- Installation complete ---


In [6]:
# Quick sanity check: can vLLM see the GPU?
import torch, vllm
print(f'torch:  {torch.__version__}')
print(f'vLLM:   {vllm.__version__}')
print(f'CUDA:   {torch.version.cuda}')
print(f'GPU:    {torch.cuda.get_device_name(0)}')
print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch:  2.10.0+cu128
vLLM:   0.19.0
CUDA:   12.8
GPU:    NVIDIA A100-SXM4-80GB
VRAM:   85.1 GB


In [7]:
# Clear previous data
import shutil, os
drive_base = '/content/drive/MyDrive/agent-repair-fever'
for d in ['outputs', 'data/processed']:
    p = os.path.join(drive_base, d)
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')
for d in ['outputs', 'data/processed']:
    p = os.path.join('/content/agent-repair', d)
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')
print('Ready for fresh run with FEVER')

Ready for fresh run with FEVER


In [26]:
!cd /content/agent-repair && git pull

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.65 KiB | 1.65 MiB/s, done.
From https://github.com/kishormorol/agent-repair
   d62fdd9..99e468c  main       -> origin/main
Updating d62fdd9..99e468c
Fast-forward
 src/env/fever_env.py | 182 ++++++++++++++++++++-------------------------------
 1 file changed, 72 insertions(+), 110 deletions(-)


In [27]:
!rm -rf /content/drive/MyDrive/agent-repair-fever/data

---
## Smoke Test (20 questions, ~10-15 min)

**Run this first** to verify everything works before committing A100 units to the full 500-question run.

In [28]:
stages = ['run_setup', 'run_generate', 'run_uncertainty', 'run_annotate',
          'run_localize', 'run_repair', 'run_eval']

for stage in stages:
    print(f'\n{"="*60}\n  {stage}\n{"="*60}')
    !python scripts/{stage}.py --config {CONFIG} --limit 20
    print()


  run_setup
15:47:35 | INFO    | setup | config=config/config_colab_fever.yaml  base=/content/drive/MyDrive/agent-repair-fever
15:47:35 | INFO    | setup | Dataset: fever (file: fever_dev.json)
15:47:35 | INFO    | setup | Downloading fever via registry download function...
  Trying: https://s3-eu-west-1.amazonaws.com/fever.public/shared_task_dev.jsonl
    failed: HTTP Error 403: Forbidden
  Trying: https://fever.ai/download/fever/shared_task_dev.jsonl
15:47:35 | INFO    | setup | Downloaded 19998 records for fever.
15:47:36 | INFO    | setup | Dataset ready: /content/drive/MyDrive/agent-repair-fever/data/raw/fever_dev.json (7.5 MB, 19998 questions)
15:47:36 | INFO    | setup | Output dirs created under: /content/drive/MyDrive/agent-repair-fever


  run_generate
15:47:36 | INFO    | stage1 | config=config/config_colab_fever.yaml  base=/content/drive/MyDrive/agent-repair-fever
15:47:36 | INFO    | stage1 | Dataset: fever (env: FEVEREnv)
15:47:36 | INFO    | stage1 | Pool: 500 questions

In [29]:
# Check smoke test produced output
import glob
tables = glob.glob('outputs/tables/*') + glob.glob('/content/drive/MyDrive/agent-repair-fever/outputs/tables/*')
figs = glob.glob('outputs/figures/*') + glob.glob('/content/drive/MyDrive/agent-repair-fever/outputs/figures/*')
print(f'Tables: {len(tables)}, Figures: {len(figs)}')
if tables or figs:
    print('Smoke test PASSED — safe to run the full pipeline below.')
else:
    print('WARNING: No outputs found. Check the logs above for errors.')

Tables: 4, Figures: 0
Smoke test PASSED — safe to run the full pipeline below.


---
## Full Pipeline (500 questions)

Only run these cells after the smoke test passes. To reset and run fresh, delete the `data/processed/` folder.

### Stage 0: Download FEVER

In [30]:
!python scripts/run_setup.py --config {CONFIG}

16:04:34 | INFO    | setup | config=config/config_colab_fever.yaml  base=/content/drive/MyDrive/agent-repair-fever
16:04:34 | INFO    | setup | Dataset: fever (file: fever_dev.json)
16:04:34 | INFO    | setup | Dataset already present: /content/drive/MyDrive/agent-repair-fever/data/raw/fever_dev.json
16:04:34 | INFO    | setup | Dataset ready: /content/drive/MyDrive/agent-repair-fever/data/raw/fever_dev.json (7.5 MB, 19998 questions)
16:04:34 | INFO    | setup | Output dirs created under: /content/drive/MyDrive/agent-repair-fever


### Stage 1: Generate ReAct Trajectories

In [31]:
!python scripts/run_generate.py --config {CONFIG}

16:04:34 | INFO    | stage1 | config=config/config_colab_fever.yaml  base=/content/drive/MyDrive/agent-repair-fever
16:04:34 | INFO    | stage1 | Dataset: fever (env: FEVEREnv)
16:04:34 | INFO    | stage1 | Pool: 500 questions
16:04:34 | INFO    | stage1 | To process: 480 of 500 (20 already done)
16:04:36 | INFO    | stage1 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-21 16:04:39.649020: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 16:04:39.724098: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorF

### Stage 2: Step-Level Uncertainty

In [32]:
!python scripts/run_uncertainty.py --config {CONFIG}

16:10:01 | INFO    | stage2 | config=config/config_colab_fever.yaml  base=/content/drive/MyDrive/agent-repair-fever
16:10:01 | INFO    | stage2 | Math metrics: 480 of 500 trajectories to do
16:10:14 | INFO    | stage2 | Math metrics done.
16:10:14 | INFO    | stage2 | Sampling metrics: 467 of 485 failed trajectories
16:10:16 | INFO    | stage2 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-21 16:10:19.317205: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 16:10:19.391762: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operati

### Stage 3: Error Annotation (72B Judge)

In [33]:
!python scripts/run_annotate.py --config {CONFIG}

16:20:54 | INFO    | stage3 | config=config/config_colab_fever.yaml  base=/content/drive/MyDrive/agent-repair-fever
16:20:54 | INFO    | stage3 | To annotate: 467 of 485 failed trajectories
16:20:56 | INFO    | stage3 | GPU VRAM: 85.094825984 GB | judge: Qwen/Qwen2.5-72B-Instruct-AWQ (85 GB -> 72B judge)
2026-07-21 16:20:59.191512: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 16:20:59.264007: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 07-21 16:21:09 [utils.py:233] non-default args: {'trust_remote_

### Stage 4: Localization Scoring

In [34]:
!python scripts/run_localize.py --config {CONFIG}

16:27:31 | INFO    | stage4 | config=config/config_colab_fever.yaml  base=/content/drive/MyDrive/agent-repair-fever
16:27:33 | INFO    | stage4 | records: 4850 (485 trajectories x 10 metrics)
                        argmax_top1  argmax_within1    mrr  threshold_hit  threshold_within1  top2_hit  top3_hit    n
metric                                                                                                               
self_consistency              0.223           0.534  0.429          0.229              0.606     0.447     0.491  485
max_token_prob_max            0.212           0.548  0.458          0.252              0.633     0.507     0.596  485
token_entropy_max             0.210           0.551  0.453          0.260              0.654     0.503     0.584  485
max_token_prob_mean           0.184           0.569  0.435          0.208              0.639     0.491     0.557  485
perplexity                    0.171           0.573  0.429          0.202              0.645     0.4

### Stage 5: Repair (126 Strategies)

In [35]:
!python scripts/run_repair.py --config {CONFIG}

16:27:34 | INFO    | stage5 | config=config/config_colab_fever.yaml  base=/content/drive/MyDrive/agent-repair-fever
16:27:34 | INFO    | stage5 | Dataset: fever (env: FEVEREnv)
16:27:34 | INFO    | stage5 | 485 failed x 126 strategies x 3 seeds = 183330 result rows (actual GPU runs are far fewer: dedup)
16:27:36 | INFO    | stage5 | GPU VRAM: 85.094825984 GB | agent model: Qwen/Qwen2.5-32B-Instruct-AWQ (85 GB -> fp16)
2026-07-21 16:27:39.645485: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-21 16:27:39.719521: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild 

### Stage 6: Evaluation

In [36]:
!python scripts/run_eval.py --config {CONFIG}

18:31:03 | INFO    | stage6 | config=config/config_colab_fever.yaml  base=/content/drive/MyDrive/agent-repair-fever
18:31:07 | INFO    | stage6 | rows: 184785 | uncertainty strategies: 120 (bt0: 60) | + ensemble

MAIN RESULTS  (fixed_% = share of FAILED trajectories that the repair fixed)
                                                           strategy  fixed_%    95%_CI  avg_tokens  avg_tool_calls  hit_true_step_%
                                                        random_step      1.2  0.7-1.7%         156            2.68             32.8
                                                       full_restart      2.8  2.0-3.7%         220            3.91             68.0
                                               oracle_targeted__bt2      3.0  2.2-3.9%         223            3.92             68.0
                                     oracle_targeted__bt2__informed      3.0  2.2-3.9%         225            3.98             68.0
                                          oracle_t

---
## Results

In [37]:
from src.utils import load_config
cfg = load_config(CONFIG)

summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())
else:
    print('Summary not yet generated. Run Stage 6 first.')

Summary not yet generated. Run Stage 6 first.


In [38]:
import pandas as pd

results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)
else:
    print('Results table not yet generated. Run Stage 6 first.')

,strategy,fixed_%,95%_CI,avg_tokens,avg_tool_calls,hit_true_step_%
0,random_step,1.2,0.7-1.7%,156,2.68,32.8
1,full_restart,2.8,2.0-3.7%,220,3.91,68.0
2,oracle_targeted__bt2,3.0,2.2-3.9%,223,3.92,68.0
3,oracle_targeted__bt2__informed,3.0,2.2-3.9%,225,3.98,68.0
4,oracle_targeted__informed,2.7,1.9-3.6%,229,4.03,100.0
...,...,...,...,...,...,...
122,unc__verbalized_confidence__topk__bt2,3.1,2.3-4.0%,208,3.68,49.3
123,unc__verbalized_confidence__topk__bt2__informed,2.7,1.9-3.5%,212,3.76,49.3
124,unc__verbalized_confidence__topk__informed,2.6,1.8-3.4%,186,3.29,29.9
125,uncertainty_ensemble_any,9.3,7.8-10.8%,10608,185.66,93.6


In [39]:
import glob
from IPython.display import display, Image

fig_dir = cfg.path('figures')
figs = sorted(glob.glob(os.path.join(fig_dir, '*.png')))
if figs:
    for f in figs:
        print(f'\n--- {os.path.basename(f)} ---')
        display(Image(filename=f, width=800))
else:
    print('No figures yet. Run Stage 6 first.')

No figures yet. Run Stage 6 first.


---
## Re-run Stages 4–6 with Cascade-Aware Strategies

**Use this after you already have stages 1–3 completed** (trajectories, uncertainty, annotations).
This pulls the latest code (with cascade rules), clears only localization/repair/eval outputs,
and re-runs stages 4→5→6 in one cell. ~3-4 hours on A100-80GB for 500 questions.

In [40]:
import os, shutil

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab_fever.yaml'

# 1. Pull latest code with cascade-aware strategies
os.chdir(REPO_DIR)
!git pull

# 2. Clear ONLY stages 4-6 outputs (keep trajectories, uncertainty, annotations)
drive_base = '/content/drive/MyDrive/agent-repair-fever'
for base in [REPO_DIR, drive_base]:
    for d in ['outputs/localization', 'outputs/repairs', 'outputs/tables', 'outputs/figures']:
        p = os.path.join(base, d)
        if os.path.exists(p):
            shutil.rmtree(p)
            print(f'Cleared {p}')

# 3. Clear checkpoint files for stages 4-6 (these live in outputs/logs/)
for base in [REPO_DIR, drive_base]:
    for ckpt in ['outputs/logs/stage4_localize.jsonl',
                 'outputs/logs/stage5_repair.jsonl',
                 'outputs/logs/stage6_eval.jsonl']:
        p = os.path.join(base, ckpt)
        if os.path.exists(p):
            os.remove(p)
            print(f'Removed checkpoint: {p}')

print('\nStages 1-3 data preserved. Ready to re-run 4→5→6.')

Already up to date.
Cleared /content/drive/MyDrive/agent-repair-fever/outputs/localization
Cleared /content/drive/MyDrive/agent-repair-fever/outputs/repairs
Cleared /content/drive/MyDrive/agent-repair-fever/outputs/tables
Cleared /content/drive/MyDrive/agent-repair-fever/outputs/figures
Removed checkpoint: /content/drive/MyDrive/agent-repair-fever/outputs/logs/stage5_repair.jsonl

Stages 1-3 data preserved. Ready to re-run 4→5→6.


In [41]:
# Run stages 4 → 5 → 6 sequentially (run this and go to sleep)
import time

for stage in ['run_localize', 'run_repair', 'run_eval']:
    t0 = time.time()
    print(f'\n{"="*60}\n  {stage}\n{"="*60}')
    !python scripts/{stage}.py --config {CONFIG}
    elapsed = (time.time() - t0) / 60
    print(f'  ✓ {stage} done in {elapsed:.1f} min\n')

print('\n' + '='*60)
print('  ALL DONE — check results below')
print('='*60)


  run_localize
18:31:27 | INFO    | stage4 | config=config/config_colab_fever.yaml  base=/content/drive/MyDrive/agent-repair-fever
18:31:29 | INFO    | stage4 | records: 4850 (485 trajectories x 10 metrics)
                        argmax_top1  argmax_within1    mrr  threshold_hit  threshold_within1  top2_hit  top3_hit    n
metric                                                                                                               
self_consistency              0.223           0.534  0.429          0.229              0.606     0.447     0.491  485
max_token_prob_max            0.212           0.548  0.458          0.252              0.633     0.507     0.596  485
token_entropy_max             0.210           0.551  0.453          0.260              0.654     0.503     0.584  485
max_token_prob_mean           0.184           0.569  0.435          0.208              0.639     0.491     0.557  485
perplexity                    0.171           0.573  0.429          0.202           

In [42]:
# View cascade-aware results
from src.utils import load_config
cfg = load_config(CONFIG)

summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())

import pandas as pd
results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)

import glob
from IPython.display import display, Image
figs = sorted(glob.glob(os.path.join(cfg.path('figures'), '*.png')))
for f in figs:
    print(f'\n--- {os.path.basename(f)} ---')
    display(Image(filename=f, width=800))

,strategy,fixed_%,95%_CI,avg_tokens,avg_tool_calls,hit_true_step_%
0,random_step,1.2,0.6-1.7%,157,2.68,32.8
1,full_restart,2.8,2.0-3.7%,221,3.90,68.0
2,oracle_targeted__bt2,3.2,2.3-4.0%,222,3.91,68.0
3,oracle_targeted__bt2__informed,2.9,2.1-3.8%,224,3.97,68.0
4,oracle_targeted__informed,2.5,1.8-3.4%,228,4.02,100.0
...,...,...,...,...,...,...
122,unc__verbalized_confidence__topk__bt2,3.4,2.5-4.3%,208,3.68,49.3
123,unc__verbalized_confidence__topk__bt2__informed,2.5,1.8-3.4%,211,3.75,49.3
124,unc__verbalized_confidence__topk__informed,2.5,1.8-3.4%,185,3.28,29.9
125,uncertainty_ensemble_any,9.6,8.1-11.1%,10600,185.46,93.6
